# Example-10: ICA filtering

In [1]:
# Import

import numpy
import torch

import sys
sys.path.append('..')

from harmonica.util import data_load
from harmonica.window import Window
from harmonica.data import Data
from harmonica.frequency import Frequency
from harmonica.filter import Filter

import matplotlib.pyplot as plt

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())

True


In [2]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [3]:
# Compute reference frequency for test TbT data

# Load TbT

length = 4096

w = Window(length, 'cosine_window', 2.0, dtype=dtype, device=device)
d = Data.from_file(54, w, '../virtual_tbt.npy', dtype=dtype, device=device)

# Compute reference frequency

d.window_remove_mean()
d.window_apply()
f = Frequency(d)
f('parabola')
d.reset()
ref = f.frequency.mean().cpu().item()
print(ref)

0.463116901262685


In [4]:
# Set parameters

length = 1024

# Set noise std for each BPM
# Note, each BPM has a different value of noise sigma

s = 1.0E-4*torch.rand(54, dtype=dtype, device=device)

# Test TbT data with random noise

w = Window(length, 'cosine_window', 1.0, dtype=dtype, device=device)
d = Data.from_file(54, w, '../virtual_tbt.npy', dtype=dtype, device=device)
d.add_noise(s)
d.data.copy_(d.work)

# Set filter instance

l = Filter(d)

# Set frequency instance

f = Frequency(d)

In [5]:
# Estimate noise (optimal SVD truncation)

_, n = l.estimate_noise(limit=64, cpu=True)

# Maximum noise estimation error

print(torch.max(100*torch.abs(n - s)/s).cpu().item())

10.47386737532945


In [6]:
# Compute frequencies without filtering

# Note, since error is random, i.e. deviation of mean from the reference
# The result should be judged mostly by frequency spread (assuming bias can be neglected)
# Depending on the noise level, windowing might (and in general does) increase spread

f = Frequency(d)

# Without window

f('parabola')
m, s = f.frequency.mean().cpu().item(), f.frequency.std().cpu().item()
print(f'error: {abs(ref - m):<16.9} spread: {s:<16.9} case: raw data without window')

# With window
# Note, in general, window will increase frequency spread, but reduce bias

d.window_remove_mean()
d.window_apply()
f('parabola')
d.reset()
m, s = f.frequency.mean().cpu().item(), f.frequency.std().cpu().item()
print(f'error: {abs(ref - m):<16.9} spread: {s:<16.9} case: raw data with window')

error: 4.01550894e-08   spread: 6.96058661e-07   case: raw data without window
error: 6.33054142e-09   spread: 8.36988268e-07   case: raw data with window


In [7]:
# Perform ICA filtering

from sklearn.decomposition import FastICA


ica = FastICA(n_components=4, max_iter=10000, tol=1.0E-6, whiten='unit-variance')
fit = torch.tensor(ica.fit_transform(d.data.T.cpu().numpy()), dtype=dtype, device=device)
mix = torch.tensor(ica.mixing_, dtype=dtype, device=device)
tbt = Data.from_data(w, mix @ fit.T)

f = Frequency(tbt)

# Without window

f('parabola')
m, s = f.frequency.mean().cpu().item(), f.frequency.std().cpu().item()
print(f'error: {abs(ref - m):<16.9} spread: {s:<16.9}')

# With window
# Note, in general, window will increase frequency spread, but reduce bias

tbt.window_remove_mean()
tbt.window_apply()
f('parabola')
tbt.reset()
m, s = f.frequency.mean().cpu().item(), f.frequency.std().cpu().item()
print(f'error: {abs(ref - m):<16.9} spread: {s:<16.9}')

error: 1.09720636e-07   spread: 4.56327957e-07  
error: 1.09525459e-07   spread: 3.4072578e-07   
